In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import implicit
import itertools
import random

c:\Users\jjacq\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv("data/split/train_split.csv")
val_df = pd.read_csv("data/split/val_split.csv")
test_df = pd.read_csv("data/split/test_split.csv")

steam = pd.read_csv("steam_store/steam.csv")
steam["genres"] = steam["genres"].fillna("").apply(lambda x: x.split(";"))

steam_small = steam[["appid", "genres"]].copy()

print("Train:", train_df.shape, "users:", train_df["user_id"].nunique())
print("Val:  ", val_df.shape,   "users:", val_df["user_id"].nunique())
print("Test: ", test_df.shape,  "users:", test_df["user_id"].nunique())


Train: (63905, 8) users: 9906
Val:   (11649, 8) users: 9906
Test:  (12784, 8) users: 9906


In [3]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0: return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)

def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0: return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)

def ndcg_at_k(rec_k, rel_set):
    if not rec_k: return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0

def hit_score_at_k(rec_k, rel_set):
    return 1.0 if any((i in rel_set) for i in rec_k) else 0.0

def map_at_k(rec_k, rel_set):
    if not rec_k:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0

def diversity_at_k(rec_k_dict, info_videojuegos):
    diversidades = []
    for uid, recs in rec_k_dict.items():
        generos = {info_videojuegos[i][1] for i in recs if i in info_videojuegos}
        diversidades.append(len(generos))
    return np.mean(diversidades) if diversidades else np.nan

In [4]:
def to_bool(x):
    if isinstance(x, str):
        return x.lower() == "true"
    return bool(x)

for df in (train_df, val_df, test_df):
    if "is_recommended" in df.columns:
        df["is_recommended"] = df["is_recommended"].apply(to_bool)
    else:
        df["is_recommended"] = False

In [5]:
train_base = train_df.copy()
train_base["hours"] = train_base["hours"].clip(lower=0.0)

# usuarios
user_cats = train_base["user_id"].astype("category")
train_base["user_idx"] = user_cats.cat.codes
idx2user = dict(enumerate(user_cats.cat.categories))
user2idx = {u: i for i, u in idx2user.items()}

# items
item_cats = train_base["app_id"].astype("category")
train_base["item_idx"] = item_cats.cat.codes
idx2item = dict(enumerate(item_cats.cat.categories))
item2idx = {i: j for j, i in idx2item.items()}

n_users = len(idx2user)
n_items = len(idx2item)

print(f"\nusuarios distintos en train (reindexados): {n_users}")
print(f"items distintos en train (reindexados):   {n_items}")


usuarios distintos en train (reindexados): 9906
items distintos en train (reindexados):   2822


In [6]:
# mapear géneros desde steam.csv
train_base["genres"] = train_base["app_id"].map(
    steam_small.set_index("appid")["genres"]
).fillna("").apply(lambda x: x if isinstance(x, list) else [])

# peso por género
train_base["genre_weight"] = train_base["genres"].apply(len).astype(float)

# índices fijos
row_idx = train_base["item_idx"].values
col_idx = train_base["user_idx"].values


In [ ]:
def build_rel_idx(df, user2idx, item2idx):
    df2 = df.copy()

    # 1) quedarnos solo con usuarios e ítems que ALS conoce
    df2 = df2[df2["user_id"].isin(user2idx.keys())]
    df2 = df2[df2["app_id"].isin(item2idx.keys())]

    df2["user_idx"] = df2["user_id"].map(user2idx)
    df2["item_idx"] = df2["app_id"].map(item2idx)

    # todos los usuarios de este split que ALS puede manejar
    all_user_idx = df2["user_idx"].unique()

    # 2) relevantes = sólo filas con is_recommended == True
    df_rel = df2[df2["is_recommended"] == True]

    rel_true = (
        df_rel
        .groupby("user_idx")["item_idx"]
        .apply(set)
        .to_dict()
    )

    # 3) diccionario final: todos los usuarios del split que ALS puede manejar,
    #con set() vacío si no tienen ningún True
    rel = {u_idx: rel_true.get(u_idx, set()) for u_idx in all_user_idx}

    print("usuarios únicos en df (raw):",
          df["user_id"].nunique())
    print("usuarios en df con user y item conocidos por ALS:",
          len(all_user_idx))
    print("suarios con al menos un is_recommended=True",
          len(rel_true))
    print("------------------------")

    return rel

val_rel_idx  = build_rel_idx(val_df,  user2idx, item2idx)
test_rel_idx = build_rel_idx(test_df, user2idx, item2idx)

print("usuarios en test_rel_idx:", len(test_rel_idx))
print("usuarios únicos en test_df:", test_df["user_id"].nunique())


usuarios únicos en df (raw): 9906
usuarios en df con user y item conocidos por ALS: 9823
suarios con al menos un is_recommended=True 8560
------------------------
usuarios únicos en df (raw): 9906
usuarios en df con user y item conocidos por ALS: 9664
suarios con al menos un is_recommended=True 8465
------------------------
usuarios en test_rel_idx: 9664
usuarios únicos en test_df: 9906


In [8]:
# nos quedamos solo con usuarios/ítems que ALS conoce
train_idx = train_df[
    train_df["user_id"].isin(user2idx.keys()) &
    train_df["app_id"].isin(item2idx.keys())
].copy()

train_idx["user_idx"] = train_idx["user_id"].map(user2idx)
train_idx["item_idx"] = train_idx["app_id"].map(item2idx)

# diccionario global: user_idx -> set(item_idx) vistos en train
user_seen = (
    train_idx
    .groupby("user_idx")["item_idx"]
    .apply(set)
    .to_dict()
)
n_items = len(item2idx) 


In [9]:
K = 10

# se hizo una ponderacion con horas, is_recommended y genero

def run_als(alpha, beta, gamma, factors, reg, iterations):

    conf_base  = 1.0 + alpha * np.log1p(train_base["hours"].values)
    rec_factor = 1.0 + beta  * train_base["is_recommended"].astype(float).values
    genre_factor = 1.0 + gamma * train_base["genre_weight"].values

    confidence = conf_base * rec_factor * genre_factor
    data = confidence.astype(np.float32)

    item_user_mat = csr_matrix(
        (data, (row_idx, col_idx)),
        shape=(n_items, n_users)
    ).tocsr()


    model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=reg,
        iterations=iterations,
        use_gpu=False
    )
    model.fit(item_user_mat)

    precs, recs, ndcgs = [], [], []

    for u_idx, rel_items in val_rel_idx.items():
        if u_idx >= model.user_factors.shape[0]:
            continue

        scores = model.user_factors[u_idx] @ model.item_factors.T
        seen = user_seen.get(u_idx, set())

        ranked = [(it, scores[it]) for it in range(n_items) if it not in seen]
        if not ranked:
            continue

        ranked.sort(key=lambda x: x[1], reverse=True)
        topk_item_idxs = [it for it, sc in ranked[:K]]

        precs.append(precision_at_k(topk_item_idxs, rel_items))
        recs.append(recall_at_k(topk_item_idxs, rel_items))
        ndcgs.append(ndcg_at_k(topk_item_idxs, rel_items))

    if not recs:
        return model, 0.0, 0.0, 0.0

    return (
        model,
        float(np.mean(precs)),
        float(np.mean(recs)),
        float(np.mean(ndcgs)),
    )


In [ ]:
alpha_grid  = [1.0, 2.0, 5.0]
beta_grid   = [0.5, 1.0, 2.0]
gamma_grid  = [0.0, 0.2, 0.5, 1.0]
factors_grid = [32, 64, 128, 256]
reg_grid     = [0.01, 0.05]
iter_grid    = [20, 50]

# random search de 30 iteraciones
param_space = list(itertools.product(
    alpha_grid,
    beta_grid,
    gamma_grid,
    factors_grid,
    reg_grid,
    iter_grid
))

n_iter = 30
random_seed = 42

random.seed(random_seed)
sampled_configs = random.sample(param_space, n_iter)

results = []
best_model = None
best_cfg = None
best_recall = -1.0

for (alpha, beta, gamma, f, reg, iters) in sampled_configs:
    print(f"alpha={alpha}, beta={beta}, gamma={gamma}, factors={f}, reg={reg}, iters={iters}...")

    model, prec, rec, ndcg = run_als(alpha, beta, gamma, f, reg, iters)

    results.append({
        "alpha": alpha,
        "beta": beta,
        "gamma": gamma,
        "factors": f,
        "reg": reg,
        "iterations": iters,
        "precision@10_val": prec,
        "recall@10_val": rec,
        "ndcg@10_val": ndcg
    })

    if rec > best_recall:
        best_recall = rec
        best_model = model
        best_cfg = (alpha, beta, gamma, f, reg, iters)

    print(f" -> P@10={prec:.4f}, R@10={rec:.4f}, NDCG={ndcg:.4f}\n")

results_df = pd.DataFrame(results).sort_values("recall@10_val", ascending=False)
results_df.head()


c:\Users\jjacq\AppData\Local\Programs\Python\Python310\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


alpha=1.0, beta=1.0, gamma=1.0, factors=32, reg=0.05, iters=20...


100%|██████████| 20/20 [00:00<00:00, 72.61it/s]


 -> P@10=0.0004, R@10=0.0032, NDCG=0.0013

alpha=1.0, beta=0.5, gamma=0.2, factors=128, reg=0.01, iters=50...


100%|██████████| 50/50 [00:01<00:00, 40.52it/s]


...

100%|██████████| 20/20 [00:00<00:00, 47.81it/s]


 -> P@10=0.0003, R@10=0.0030, NDCG=0.0012

alpha=1.0, beta=1.0, gamma=0.2, factors=256, reg=0.05, iters=20...


100%|██████████| 20/20 [00:11<00:00,  1.74it/s]


 -> P@10=0.0004, R@10=0.0034, NDCG=0.0016



,alpha,beta,gamma,factors,reg,iterations,precision@10_val,recall@10_val,ndcg@10_val
3,2.0,0.5,1.0,128,0.05,20,0.000536,0.004898,0.001803
11,1.0,0.5,0.2,256,0.05,20,0.000501,0.004827,0.001858
16,1.0,0.5,0.2,128,0.05,50,0.000501,0.004827,0.001766
4,2.0,0.5,0.5,64,0.01,20,0.000465,0.004648,0.001922
8,1.0,1.0,0.2,128,0.01,50,0.000465,0.004469,0.001553


In [11]:
print("\nmejor config:")
print(f"alpha={best_cfg[0]}, beta={best_cfg[1]}, gamma={best_cfg[2]}, "
      f"factors={best_cfg[3]}, reg={best_cfg[4]}, iters={best_cfg[5]}")
print(f"Recall@10_val={best_recall:.4f}")


mejor config:
alpha=2.0, beta=0.5, gamma=1.0, factors=128, reg=0.05, iters=20
Recall@10_val=0.0049


In [ ]:
K = 10

def evaluate_model_on_split(model, rel_idx, user_seen, k=10):

    precs, recs, ndcgs, hits, maps = [], [], [], [], []
    rec_k_dict = {}

    total_users = len(rel_idx)
    skipped_out_of_range = 0
    skipped_no_ranked = 0

    for u_idx, rel_items in rel_idx.items():
        # 1) usuario fuera de rango (modelo no tiene factores para él)
        if u_idx >= model.user_factors.shape[0]:
            skipped_out_of_range += 1
            continue

        # puntajes para todos los ítems
        scores = model.user_factors[u_idx] @ model.item_factors.T

        # filtrar ítems ya vistos
        seen = user_seen.get(u_idx, set())
        ranked = [(it, scores[it]) for it in range(n_items) if it not in seen]

        # 2) sin candidatos (ha visto todo o algo raro)
        if not ranked:
            skipped_no_ranked += 1
            continue

        ranked.sort(key=lambda x: x[1], reverse=True)
        topk = [it for it, s in ranked[:k]]

        # guardar para diversidad
        rec_k_dict[u_idx] = topk

        # metricas por usuario
        precs.append(precision_at_k(topk, rel_items))
        recs.append(recall_at_k(topk, rel_items))
        ndcgs.append(ndcg_at_k(topk, rel_items))
        hits.append(hit_score_at_k(topk, rel_items))
        maps.append(map_at_k(topk, rel_items))

    n_eval = len(rec_k_dict)

    print("---- evaluate_model_on_split ----")
    print("usuarios en rel_idx:", total_users)
    print("saltados por user_factors (out rng):", skipped_out_of_range)
    print("saltados por ranked vacío:", skipped_no_ranked)
    print("usuarios efectivamente evaluados:", n_eval)
    print("------------------------------------")

    return (
        np.mean(precs) if precs else np.nan,
        np.mean(recs)  if recs  else np.nan,
        np.mean(ndcgs) if ndcgs else np.nan,
        np.mean(hits)  if hits  else np.nan,
        np.mean(maps)  if maps  else np.nan,
        rec_k_dict
    )

prec_test, rec_test, ndcg_test, hit_test, map_test, rec_k_dict = evaluate_model_on_split(
    best_model,
    test_rel_idx,   # construido con build_rel_idx usando is_recommended=True
    user_seen,
    k=K
)

# usuarios que había en test_rel_idx
n_users_test = len(test_rel_idx)

# usuarios que se evaluaron efectivamente (tienen ranking en rec_k_dict)
n_eval_test = len(rec_k_dict)

print(f"\nusuarios en test_rel_idx:        {n_users_test}")
print(f"usuarios efectivamente evaluados {n_eval_test}")

info_videojuegos = {}
for idx, app in idx2item.items():
    genres_series = steam_small[steam_small["appid"] == app]["genres"]
    if len(genres_series) > 0:
        lista = genres_series.values[0]
        genero_principal = lista[0] if len(lista) > 0 else "desconocido"
    else:
        genero_principal = "Unknown"
    info_videojuegos[idx] = (app, genero_principal)

div_test = diversity_at_k(rec_k_dict, info_videojuegos)



---- evaluate_model_on_split ----
usuarios en rel_idx: 9664
saltados por user_factors (out rng): 6913
saltados por ranked vacío: 0
usuarios efectivamente evaluados: 2751
------------------------------------

usuarios en test_rel_idx:        9664
usuarios efectivamente evaluados 2751


In [15]:
print(f"usuarios en test_rel_idx: {n_users_test}")
print(f"usuarios evaluados por ALS: {n_eval_test}")
print(f"precision@{K}:{prec_test:.6f}")
print(f"recall@{K}: {rec_test:.6f}")
print(f"F1@{K}: {2 * prec_test * rec_test / (prec_test + rec_test + 1e-10):.6f}")
print(f"NDCG@{K}: {ndcg_test:.6f}")
print(f"Hit@{K}: {hit_test:.6f}")
print(f"MAP@{K}: {map_test:.6f}")
print(f"Diversity@{K}: {div_test:.6f}")


print("mejor configuración encontrada:")
print(f"alpha={best_cfg[0]}, beta={best_cfg[1]}, gamma={best_cfg[2]}, "
      f"factors={best_cfg[3]}, reg={best_cfg[4]}, iters={best_cfg[5]}")

usuarios en test_rel_idx: 9664
usuarios evaluados por ALS: 2751
precision@10:0.000327
recall@10: 0.003029
F1@10: 0.000591
NDCG@10: 0.001184
Hit@10: 0.003272
MAP@10: 0.000619
Diversity@10: 4.286805
mejor configuración encontrada:
alpha=2.0, beta=0.5, gamma=1.0, factors=128, reg=0.05, iters=20
